In [ ]:
import pandas as pd
import time
import os
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login

# ==========================================
# 1. KİMLİK DOĞRULAMA (Hugging Face)
# ==========================================
# Gemma ve diğer kapalı modeller için yetkilendirme
login(token="XXXX")

# ==========================================
# 2. DONANIM VE PERFORMANS METRİKLERİ
# ==========================================
def get_model_size_mb(model):
    """Modelin parametre ve buffer boyutunu MB cinsinden hesaplar."""
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    return (param_size + buffer_size) / 1024**2

def get_vram_usage():
    """Çalışma anında GPU'da ayrılan maksimum VRAM miktarını MB cinsinden döner."""
    return torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else 0

# ==========================================
# 3. MODEL VE VERİ TANIMLAMALARI
# ==========================================
models_to_test = {
    "SmolLM2": "HuggingFaceTB/SmolLM2-360M-Instruct",
    "TinyLlama": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "Qwen2.5": "Qwen/Qwen2.5-1.5B-Instruct",
    "Gemma2B": "google/gemma-2-2b-it"
}

# Test verisini oku
data_path = "drive/MyDrive/LLM-PROJE/test.csv"
if not os.path.exists(data_path):
    raise FileNotFoundError(f"Test verisi bulunamadı! Lütfen şu yolu kontrol et: {data_path}")

test_df = pd.read_csv(data_path)

# ==========================================
# 4. ANA BENCHMARK DÖNGÜSÜ
# ==========================================
for model_name, model_id in models_to_test.items():
    print(f"\n" + "="*40)
    print(f"--- {model_name} Test Süreci Başlatıldı ---")
    print("="*40)

    # Her model geçişinde RAM/VRAM'i tamamen temizle (Colab çökmesini önler)
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    # Tokenizer yükleme ve konfigürasyon
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Modeli GPU (mümkünse) üzerinde float16 hassasiyetinde yükle
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        torch_dtype=torch.float16
    )

    # Donanım profilini çıkar
    model_size = get_model_size_mb(model)
    print(f"-> Model Boyutu: {model_size:.2f} MB")

    results = []

    # Her satır için çıkarım yap (Inference Loop)
    for index, row in test_df.iterrows():
        text = row['İcerik']
        true_label = row['Kategori']

        messages = [
            {
                "role": "user",
                "content": f"Talimat: Sen bir İş Hukuku uzmanısın. Aşağıdaki içeriği oku ve sadece ilgili kategorisini tek kelimeyle yaz. Açıklama yapma.\n\nİçerik: {text}"
            }
        ]

        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        start_time = time.time()
        with torch.no_grad():
            outputs = model.generate(
                inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=15,
                pad_token_id=tokenizer.pad_token_id,
                do_sample=False  # Kararlı ve tekrarlanabilir çıktılar için deterministik mod
            )
        end_time = time.time()

        # Sadece modelin yeni ürettiği yanıt token'larını kesip alalım
        generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
        prediction = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

        results.append({
            "Soru": text,
            "Gerçek_Etiket": true_label,
            "Tahmin": prediction,
            "Latency_ms": (end_time - start_time) * 1000,
            "VRAM_Usage_MB": get_vram_usage(),
            "Model_Size_MB": model_size
        })

    # ==========================================
    # 5. METRİKLERİ VE SONUÇLARI KAYDETME
    # ==========================================
    res_df = pd.DataFrame(results)
    save_filename = f"drive/MyDrive/LLM-PROJE/zero_shot_{model_name}_final.csv"

    res_df.to_csv(save_filename, index=False, encoding='utf-8-sig')
    print(f"\n[BAŞARILI] {model_name} tamamlandı. Sonuçlar '{save_filename}' adresine kaydedildi.")

print("\n" + "="*40)
print("TÜM MODELLERİN ZERO-SHOT TESTLERİ BAŞARIYLA BİTTİ!")
print("="*40)